In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

OUT_DIR = "eda_images"
os.makedirs(OUT_DIR, exist_ok=True)
df = pd.read_csv("/content/drive/MyDrive/StackOverFlow/survey_cleaned_base.csv")

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 14
})

if "ConvertedCompYearly" in df.columns:
    sal = pd.to_numeric(df["ConvertedCompYearly"], errors="coerce").dropna()
    sal_plot = sal[(sal >= 1000) & (sal <= 500_000)]
    sal_zoom = sal[(sal >= 1000) & (sal <= 200_000)]

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    axes[0,0].hist(sal_plot, bins=500, color="#5B9BD5")
    axes[0,0].set_title("(а) Гистограмма, 500 бинов")
    axes[0,1].hist(sal_zoom, bins=400, color="#ED7D31")
    for x in [25_000, 50_000, 75_000, 100_000, 120_000, 150_000]:
        axes[0,1].axvline(x, color="red", linestyle="--", alpha=0.45)
    axes[0,1].set_title("(б) Зум 0–200k")
    axes[1,0].hist(np.log10(sal[sal > 0]), bins=100, color="#5B9BD5")
    axes[1,0].set_title("(в) log10(годового дохода)")
    x_ecdf = np.sort(sal_plot.values)
    axes[1,1].plot(x_ecdf, np.arange(1, len(x_ecdf)+1)/len(x_ecdf), color="#5B9BD5")
    axes[1,1].set_xlim(0, 300_000)
    axes[1,1].set_title("(г) Эмпирическая функция распределения")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "11_salary_detailed.png"), dpi=300)
    plt.close(fig)

    sal_int = sal[sal >= 1000].astype(np.int64)
    first1 = sal_int.astype(str).str[0].astype(int)
    d1 = np.arange(1, 10)
    obs1 = first1.value_counts(normalize=True).sort_index().reindex(d1).fillna(0)
    benf1 = np.log10(1 + 1/d1)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(d1, obs1.values, color="#5B9BD5", alpha=0.75, label="Наблюдаемая частота")
    ax.plot(d1, benf1, color="red", marker="o", linewidth=2, label="Закон Бенфорда")
    ax.set_xticks(d1)
    ax.set_title("Распределение первой цифры годового дохода")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "12a_benford_first1.png"), dpi=300)
    plt.close(fig)

    first2 = sal_int.astype(str).str[:2].astype(int)
    first2 = first2[(first2 >= 10) & (first2 <= 99)]
    d2 = np.arange(10, 100)
    counts2 = first2.value_counts().sort_index().reindex(d2).fillna(0)
    obs2 = counts2 / counts2.sum()
    benf2 = np.log10(1 + 1/d2)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(d2, obs2.values, color="#5B9BD5", alpha=0.75, width=0.85)
    ax.plot(d2, benf2, color="red", linewidth=1.8, marker="o", markersize=3)
    ax.set_xticks(np.arange(10, 100, 5))
    ax.set_title("Распределение первых двух цифр годового дохода")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "12b_benford_first2.png"), dpi=300)
    plt.close(fig)

    last2_counts = (sal_int % 100).value_counts().sort_index().reindex(np.arange(0, 100)).fillna(0)
    obs_last = last2_counts / last2_counts.sum()

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw=dict(width_ratios=[1, 1], wspace=0.18))
    axL.bar(np.arange(100), obs_last.values, color="#ED7D31", width=0.85)
    axL.axhline(0.01, color="red", linestyle="--")
    axL.set_xlim(-1, 100)
    axL.set_title("(а) Полный вид")

    axR.bar(np.arange(100), obs_last.values, color="#ED7D31", width=0.85)
    axR.axhline(0.01, color="red", linestyle="--")
    axR.set_ylim(0, 0.03)
    axR.set_xlim(-1, 100)
    axR.set_title("(б) Зум 0–0.03")
    fig.suptitle("Распределение последних двух цифр годового дохода")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "12c_last_two_digits.png"), dpi=300)
    plt.close(fig)

/tmp/ipykernel_2313/1731827377.py:90: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
